# Martini: Membrane Melting Temperature Optimization

This notebook optimizes Martini force-field parameters so that the fitted membrane melting temperature matches an experimental target, using DiffTRe with multiple GROMACS simulations at different temperatures coordinated via Ray.

## What is mythos?

**[Mythos](https://github.com/mythos-bio/mythos)** is a differentiable molecular simulation framework for parameterizing coarse-grained force fields. It provides automatic differentiation through simulation workflows — enabling gradient-based optimization of force field parameters to match experimental or all-atom reference data.

This notebook demonstrates melting temperature optimization for lipid membranes. We run GROMACS simulations across a temperature range, compute the melting temperature from area-per-lipid profiles via sigmoid fitting, and use DiffTRe to optimize the force-field parameters. DiffTRe computes per-state Boltzmann weights using the per-state temperature stored on each `SimulatorTrajectory`.

## Imports

In [ ]:
import logging
from pathlib import Path

import jax
import MDAnalysis
import optax
from jax import numpy as jnp
from mythos.energy.base import ComposedEnergyFunction
from mythos.energy.martini.base import MartiniTopology
from mythos.energy.martini.m2 import (
    Angle,
    AngleConfiguration,
    Bond,
    BondConfiguration,
)
from mythos.input.gromacs_input import read_params_from_topology
from mythos.observables.membrane_melting_temp import MembraneMeltingTemp
from mythos.optimization.objective import DiffTReObjective
from mythos.optimization.optimization import OptimizerOutput, RayOptimizer
from mythos.simulators.gromacs.gromacs import KB, GromacsSimulator
from mythos.simulators.gromacs.utils import preprocess_topology
from mythos.ui.loggers.console import ConsoleLogger

jax.config.update("jax_enable_x64", True)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## Configuration

In [ ]:
INPUT_DIR = Path("../../data/templates/martini/m2/DMPC/273K").resolve()  # adjust to your input dir
TARGET_TM = 314.0  # target melting temperature in K
LIPID_SEL = "name GL1 GL2"
OPT_STEPS = 50
LEARNING_RATE = 1e-3
SIMULATION_STEPS = 500_000
EQUILIBRATION_STEPS = 100_000
SNAPSHOT_STEPS = 10_000
GROMACS_BINARY = None  # uses gmx on PATH

# Simulation temperatures (K)
SIM_TEMPS = (
    291, 292.5, 294, 295.5, 297, 298.5, 300, 301.5,
    303, 304.5, 306, 307.5, 309, 310.5, 312, 313.5,
    315, 316.5, 318, 319.5, 321, 322.5, 324, 325.5,
    327, 328.5, 330, 331.5,
)

## Preprocess topology and build energy function

To build energy functions for martini we need to read parameters from a
preprocessed topology file (which expands any macros and includes all files).
Likewise we need to read the compiled topology which comes in the .tpr format
read into a mythose `MartiniTopology` object via MDAnalysis.

In [ ]:
preprocessed_top = INPUT_DIR / "preprocessed.top"
preprocessed_tpr = INPUT_DIR / "preprocessed.tpr"

if not preprocessed_top.exists() or not preprocessed_tpr.exists():
    preprocess_topology(INPUT_DIR, gromacs_binary=GROMACS_BINARY)

universe = MDAnalysis.Universe(preprocessed_tpr)
universe.transfer_to_memory()
params = read_params_from_topology(preprocessed_top)

top = MartiniTopology.from_universe(universe)
energy_fn = ComposedEnergyFunction(energy_fns=[
    Bond.from_topology(topology=top, params=BondConfiguration(**params["bond_params"])),
    Angle.from_topology(topology=top, params=AngleConfiguration(**params["angle_params"])),
])

## Create simulators — one per temperature

Given access to a high level of compute-parallelism, one could further benefit
by running multiple replicas per temperature. The observable and optimization
calls would not need modification, as the optimizer handles concatenation and
the observable uses metadata for appropriate segmentation.

In [ ]:
simulators = [
    GromacsSimulator(
        name=f"sim.{i}.temp_{temp:.1f}K",
        input_dir=INPUT_DIR,
        energy_fn=energy_fn,
        simulation_steps=SIMULATION_STEPS,
        equilibration_steps=EQUILIBRATION_STEPS,
        input_overrides={"nstxout": SNAPSHOT_STEPS, "ref-t": temp},
        binary_path=GROMACS_BINARY,
    )
    for i, temp in enumerate(SIM_TEMPS)
]

## Build melting temperature observable, loss, and objective

The melting temperature observable takes care of reading the temperature from
trajectory metadata and segmenting per-temperature to compute expected APL in
those segments for fitting a sigmoid model of the form:

```math
    \text{APL}(T) = \text{apl}_0 + c_{pg} \cdot T
        + \frac{\Delta\text{APL}}{1 + \exp(-k (T - T_m))}
```

**Note**: In order to be compatible with `jax` gradient tracing, the observable
requires to know the temperatures to segment ahead of time. This introduces
potential for mismatch or for numerical instability to introduce errors. The
observable supports a tolerance check and will raise an exception if no frames
are found for a given temperature within the specified tolerance.


In [ ]:
tm_observable = MembraneMeltingTemp(
    topology=universe,
    lipid_sel=LIPID_SEL,
    temperatures=jnp.array(SIM_TEMPS) * KB,
)


def tm_loss(traj, weights, *_):
    tm = tm_observable(traj, weights=weights)
    loss = (tm - TARGET_TM * KB) ** 2  # observable reports Tm in simulation units
    return loss, (("tm", tm), ())


objective = DiffTReObjective(
    energy_fn=energy_fn,
    grad_or_loss_fn=tm_loss,
    required_observables=[name for sim in simulators for name in sim.exposes()],
    logging_observables=("loss", "tm", "neff"),
    name="MeltingTemp",
    max_valid_opt_steps=5,
)

## Run the optimization

The callback adds a human-readable melting temperature in Kelvin to the
observables at each step. As we have multiple simulators, we use the
RayOptimizer to run them in parallel, which requires specifying an
`aggregate_grad_fn` to combine the gradients from each simulator into a single
gradient for the optimizer step. Here, since we have a single objective, we can
simply take the first gradient.

In [ ]:
opt = RayOptimizer(
    simulators=simulators,
    objectives=[objective],
    optimizer=optax.adam(learning_rate=LEARNING_RATE),
    aggregate_grad_fn=lambda grads: grads[0],
    logger=ConsoleLogger(),
)

opt_params = energy_fn.opt_params()

print("\n=== Melting Temperature Optimisation ===")
print(f"  Target Tm:        {TARGET_TM} K")
print(f"  Sim temperatures: {len(SIM_TEMPS)} ({SIM_TEMPS[0]}-{SIM_TEMPS[-1]} K)")
print(f"  Parallel sims:    {len(simulators)}")
print(f"  Opt steps:        {OPT_STEPS}")
print(f"  Learning rate:    {LEARNING_RATE}")
print(f"  Parameters:       {len(opt_params)}\n")


def opt_callback(optimizer_output: OptimizerOutput, step: int):
    melting = optimizer_output.observables["MeltingTemp"]
    melting["tm_k"] = melting["tm"] / KB
    return optimizer_output, True  # return True to continue optimization, False to stop


output = opt.run(params=opt_params, n_steps=OPT_STEPS, callback=opt_callback)

print("\n=== Optimisation Complete ===")
print(f"  Final params: {output.opt_params}")